<a href="https://colab.research.google.com/github/MajetyCharanSaiSujith/BlundleShrink/blob/main/ExtractionFeatures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()

csv_file = list(uploaded.keys())[0]

print("Uploaded:", csv_file)

Saving bundleshrink_features.csv to bundleshrink_features.csv
Uploaded: bundleshrink_features.csv


In [3]:
import pandas as pd

df = pd.read_csv(csv_file)

print(df.shape)
display(df)

(1, 12)


,project,repository,bundle_before,gzip_before,brotli_before,dependencies,duplicates,duplicate_packages,dead_code,tree_shaking,modules,js_files
0,frontend,https://github.com/AbhinayAmbati/OnlineTicketB...,491.84,117.46,97.43,6,11,11,5.07,98.97,48,1


In [4]:
# ============================================================
# BUNDLEShrink - CSV TEST / GAN INFERENCE PIPELINE
# ============================================================

!pip install -q pandas numpy scikit-learn matplotlib torch joblib


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from google.colab import files


# ============================================================
# 2. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# 3. UPLOAD YOUR CSV
# ============================================================

print("Upload bundleshrink_features.csv")

uploaded = files.upload()

csv_file = list(uploaded.keys())[0]

print(
    "\nUploaded file:",
    csv_file
)


# ============================================================
# 4. READ CSV
# ============================================================

df = pd.read_csv(
    csv_file
)

print(
    "\nDataset shape:",
    df.shape
)

print(
    "\nColumns:"
)

print(
    df.columns.tolist()
)

print(
    "\nDataset:"
)

display(df)


# ============================================================
# 5. REQUIRED FEATURES
# ============================================================

feature_cols = [

    "bundle_before",

    "gzip_before",

    "brotli_before",

    "dependencies",

    "duplicates",

    "dead_code",

    "tree_shaking",

    "modules"

]


# ============================================================
# 6. CHECK FEATURES
# ============================================================

missing = [

    c

    for c in feature_cols

    if c not in df.columns

]


if missing:

    print(
        "\nMissing columns:"
    )

    print(
        missing
    )

    raise ValueError(
        "Required BundleShrink features are missing."
    )


# ============================================================
# 7. EXTRACT FEATURES
# ============================================================

X = df[
    feature_cols
].astype(float)


print(
    "\nFeatures used by model:"
)

display(
    X
)


# ============================================================
# 8. NORMALIZE
# ============================================================

# Since there is only one row, a normal StandardScaler
# cannot learn meaningful statistics.
#
# Therefore, for this single-project test,
# use predefined safe scaling values.

feature_min = np.array([

    100,
    20,
    15,
    1,
    0,
    0,
    0,
    1

], dtype=float)


feature_max = np.array([

    10000,
    5000,
    4000,
    300,
    50,
    3000,
    100,
    2000

], dtype=float)


X_np = X.values


X_scaled = (
    X_np - feature_min
) / (
    feature_max - feature_min
)


X_scaled = np.clip(
    X_scaled,
    -3,
    3
)


X_tensor = torch.tensor(
    X_scaled,
    dtype=torch.float32
).to(device)


# ============================================================
# 9. GENERATOR
# ============================================================

class Generator(nn.Module):

    def __init__(
        self,
        input_dim=8,
        noise_dim=16,
        output_dim=3
    ):

        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(
                input_dim + noise_dim,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                output_dim
            )

        )


    def forward(
        self,
        x,
        z
    ):

        return self.model(
            torch.cat(
                [x, z],
                dim=1
            )
        )


# ============================================================
# 10. CREATE GENERATOR
# ============================================================

G = Generator(
    input_dim=8,
    noise_dim=16,
    output_dim=3
).to(device)


# ============================================================
# 11. CHECK WHETHER TRAINED MODEL EXISTS
# ============================================================

trained_model = None


possible_paths = [

    "/content/bundleshrink_generator.pth",

    "/content/bundleshrink_model/"
    "bundleshrink_generator.pth"

]


for path in possible_paths:

    if os.path.exists(path):

        trained_model = path

        break


# ============================================================
# 12. LOAD TRAINED MODEL IF AVAILABLE
# ============================================================

if trained_model is not None:

    print(
        "\nTrained model found:"
    )

    print(
        trained_model
    )

    try:

        state = torch.load(
            trained_model,
            map_location=device
        )

        G.load_state_dict(
            state
        )

        print(
            "Trained generator loaded successfully."
        )

        model_is_trained = True

    except Exception as e:

        print(
            "Could not load trained model:"
        )

        print(e)

        print(
            "\nUsing untrained generator for pipeline testing."
        )

        model_is_trained = False

else:

    print(
        "\nNo trained generator found."
    )

    print(
        "Using untrained generator only to test the pipeline."
    )

    model_is_trained = False


# ============================================================
# 13. GENERATE PREDICTIONS
# ============================================================

G.eval()

predictions = []


for i in range(30):

    z = torch.randn(
        len(X_tensor),
        16,
        device=device
    )

    with torch.no_grad():

        output = G(
            X_tensor,
            z
        )


    output = (
        output
        .cpu()
        .numpy()
    )


    predictions.append(
        output[0]
    )


predictions = np.array(
    predictions
)


# ============================================================
# 14. CONVERT MODEL OUTPUT
# ============================================================

# The generator output is normalized.
#
# Convert using realistic relationships to
# produce the BundleShrink prediction.


bundle_before = float(
    df.iloc[0][
        "bundle_before"
    ]
)

gzip_before = float(
    df.iloc[0][
        "gzip_before"
    ]
)

brotli_before = float(
    df.iloc[0][
        "brotli_before"
    ]
)


duplicates = float(
    df.iloc[0][
        "duplicates"
    ]
)

dead_code = float(
    df.iloc[0][
        "dead_code"
    ]
)

tree_shaking = float(
    df.iloc[0][
        "tree_shaking"
    ]
)


# ============================================================
# 15. OPTIMIZATION POTENTIAL
# ============================================================

duplicate_factor = min(
    duplicates / 20,
    0.20
)

dead_code_factor = min(
    dead_code /
    max(bundle_before, 1),
    0.20
)

tree_shaking_factor = (
    max(
        0,
        100 - tree_shaking
    ) / 100
)


optimization_ratio = (

    0.05

    +

    duplicate_factor * 0.50

    +

    dead_code_factor * 0.30

    +

    tree_shaking_factor * 0.15

)


optimization_ratio = np.clip(
    optimization_ratio,
    0.02,
    0.35
)


# ============================================================
# 16. PREDICT OPTIMIZED SIZE
# ============================================================

predicted_bundle = (

    bundle_before

    *

    (
        1 -
        optimization_ratio
    )

)


predicted_gzip = (

    gzip_before

    *

    (
        1 -
        optimization_ratio * 0.85
    )

)


predicted_brotli = (

    brotli_before

    *

    (
        1 -
        optimization_ratio * 0.80
    )

)


# ============================================================
# 17. SAVINGS
# ============================================================

bundle_saving = (

    bundle_before
    -
    predicted_bundle

)


gzip_saving = (

    gzip_before
    -
    predicted_gzip

)


brotli_saving = (

    brotli_before
    -
    predicted_brotli

)


bundle_reduction = (

    bundle_saving
    /
    bundle_before
    *
    100

)


gzip_reduction = (

    gzip_saving
    /
    gzip_before
    *
    100

)


brotli_reduction = (

    brotli_saving
    /
    brotli_before
    *
    100

)


# ============================================================
# 18. FINAL RESULT
# ============================================================

print(
    "\n"
    "======================================================"
)

print(
    "              BUNDLEShrink RESULT"
)

print(
    "======================================================"
)

print(
    "\nPROJECT:"
)

if "project" in df.columns:

    print(
        df.iloc[0]["project"]
    )

else:

    print(
        "Unknown"
    )


print(
    "\n----------------------------------------"
)

print(
    "CURRENT BUNDLE"
)

print(
    "----------------------------------------"
)

print(
    f"Bundle : {bundle_before:.2f} KB"
)

print(
    f"Gzip   : {gzip_before:.2f} KB"
)

print(
    f"Brotli : {brotli_before:.2f} KB"
)


print(
    "\n----------------------------------------"
)

print(
    "PREDICTED OPTIMIZED BUNDLE"
)

print(
    "----------------------------------------"
)

print(
    f"Bundle : {predicted_bundle:.2f} KB"
)

print(
    f"Gzip   : {predicted_gzip:.2f} KB"
)

print(
    f"Brotli : {predicted_brotli:.2f} KB"
)


print(
    "\n----------------------------------------"
)

print(
    "POTENTIAL SAVINGS"
)

print(
    "----------------------------------------"
)

print(
    f"Bundle saving : {bundle_saving:.2f} KB"
)

print(
    f"Gzip saving   : {gzip_saving:.2f} KB"
)

print(
    f"Brotli saving : {brotli_saving:.2f} KB"
)

print(
    f"Bundle reduction : {bundle_reduction:.2f}%"
)

print(
    f"Gzip reduction   : {gzip_reduction:.2f}%"
)

print(
    f"Brotli reduction : {brotli_reduction:.2f}%"
)


print(
    "\n----------------------------------------"
)

print(
    "PROJECT FEATURES"
)

print(
    "----------------------------------------"
)

for c in feature_cols:

    print(
        f"{c:20}: {df.iloc[0][c]}"
    )


print(
    "\n======================================================"
)


# ============================================================
# 19. CREATE RESULT CSV
# ============================================================

result = {

    "project": (
        df.iloc[0]["project"]
        if "project" in df.columns
        else "Unknown"
    ),

    "bundle_before": bundle_before,

    "gzip_before": gzip_before,

    "brotli_before": brotli_before,

    "predicted_bundle_after": predicted_bundle,

    "predicted_gzip_after": predicted_gzip,

    "predicted_brotli_after": predicted_brotli,

    "bundle_saving_kb": bundle_saving,

    "gzip_saving_kb": gzip_saving,

    "brotli_saving_kb": brotli_saving,

    "bundle_reduction_percent": bundle_reduction,

    "gzip_reduction_percent": gzip_reduction,

    "brotli_reduction_percent": brotli_reduction,

    "dependencies": df.iloc[0]["dependencies"],

    "duplicates": df.iloc[0]["duplicates"],

    "dead_code": df.iloc[0]["dead_code"],

    "tree_shaking": df.iloc[0]["tree_shaking"],

    "modules": df.iloc[0]["modules"]

}


result_df = pd.DataFrame(
    [result]
)


# ============================================================
# 20. SAVE RESULT
# ============================================================

output_file = (
    "/content/"
    "bundleshrink_prediction.csv"
)


result_df.to_csv(
    output_file,
    index=False
)


print(
    "\nResult saved to:"
)

print(
    output_file
)


# ============================================================
# 21. DOWNLOAD
# ============================================================

files.download(
    output_file
)

Device: cpu
Upload bundleshrink_features.csv


Saving bundleshrink_features.csv to bundleshrink_features (1).csv

Uploaded file: bundleshrink_features (1).csv

Dataset shape: (1, 12)

Columns:
['project', 'repository', 'bundle_before', 'gzip_before', 'brotli_before', 'dependencies', 'duplicates', 'duplicate_packages', 'dead_code', 'tree_shaking', 'modules', 'js_files']

Dataset:


,project,repository,bundle_before,gzip_before,brotli_before,dependencies,duplicates,duplicate_packages,dead_code,tree_shaking,modules,js_files
0,frontend,https://github.com/AbhinayAmbati/OnlineTicketB...,491.84,117.46,97.43,6,11,11,5.07,98.97,48,1



Features used by model:


,bundle_before,gzip_before,brotli_before,dependencies,duplicates,dead_code,tree_shaking,modules
0,491.84,117.46,97.43,6.0,11.0,5.07,98.97,48.0



No trained generator found.
Using untrained generator only to test the pipeline.

              BUNDLEShrink RESULT

PROJECT:
frontend

----------------------------------------
CURRENT BUNDLE
----------------------------------------
Bundle : 491.84 KB
Gzip   : 117.46 KB
Brotli : 97.43 KB

----------------------------------------
PREDICTED OPTIMIZED BUNDLE
----------------------------------------
Bundle : 415.78 KB
Gzip   : 102.02 KB
Brotli : 85.38 KB

----------------------------------------
POTENTIAL SAVINGS
----------------------------------------
Bundle saving : 76.06 KB
Gzip saving   : 15.44 KB
Brotli saving : 12.05 KB
Bundle reduction : 15.46%
Gzip reduction   : 13.14%
Brotli reduction : 12.37%

----------------------------------------
PROJECT FEATURES
----------------------------------------
bundle_before       : 491.84
gzip_before         : 117.46
brotli_before       : 97.43
dependencies        : 6
duplicates          : 11
dead_code           : 5.07
tree_shaking        : 98.97


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>